## Isolated ensemble-consensus experiment

This notebook writes all results below `outputs/` and does not overwrite the
validated `nature`-reference experiment. Shared paths are defined in
`project_paths.py`.

In [ ]:
import importlib
import project_paths as _project_paths
_project_paths = importlib.reload(_project_paths)
from project_paths import (
    PROJECT_ROOT, ANALYSIS_DATA_DIR, OSSE_DATA_DIR,
    cle_target_experiment_paths, cse_experiment_paths, ensure_output_directories,
)
ensure_output_directories()
EXPERIMENT_NAME = 'xyz_control_ensemble_consensus_qiwenevents'
EXPERIMENT_ROOT, RESULTS_DIR, EXPERIMENT_FIGURES_DIR = cse_experiment_paths(EXPERIMENT_NAME)
CLE_TARGET_EXPERIMENT_ROOT, CLE_TARGET_DATA_DIR, CLE_TARGET_FIGURES_DIR = cle_target_experiment_paths(EXPERIMENT_NAME)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CLE_TARGET_DATA_DIR.mkdir(parents=True, exist_ok=True)
CLE_TARGET_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Qiwen CSE xyz Control with an Ensemble-Consensus Regime Reference

This experiment changes only the source used to define the current Lorenz
regime. The sign is inferred from the complete analysis ensemble, following
Takemasa's proposal. No truth information is used by the controller.

## Imports


In [ ]:
import importlib
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from class_da_sys import da_system
from class_Lorenz63 import run_RK4
from class_cse import plot_controlled_vs_free_nature_run
import class_cse_qiwen as cse_qiwen
cse_qiwen = importlib.reload(cse_qiwen)
from class_cse_qiwen import (
    build_and_apply_perturbation,
    detect_regime_changes,
    find_past_safe_trajectory,
    integrate_and_align,
    lorenz63,
    plot_found_past_safe_trajectory,
    plot_cse_initialization,
    plot_failed_event_diagnostic,
    plot_3d_nature_run_comparison,
    plot_lorenz_attractor_aligned_cse_starts,
    plot_regime_crossings,
    run_single_cse as run_single_cse_core,
    run_single_cse_mini_test,
    summarize_cse_batch_results,
    summarize_cse_batch_results_by_D,
    run_perturbation_unit_test,
)


## DA System Setup


In [ ]:
BASE_PATH = PROJECT_ROOT
PathFigures = EXPERIMENT_FIGURES_DIR
PathFigures.mkdir(parents=True, exist_ok=True)

# Full-state observations and full xyz actuation are retained from notebook 6.
# The only experimental change is the current-regime definition.
CONTROL_INDICES = None
CONTROL_LABEL = 'xyz'
REGIME_REFERENCE_MODE = 'ensemble_consensus'
AMBIGUOUS_REGIME_POLICY = 'stop'
CANDIDATE_POOL = 'full'
EVENT_SELECTION = 'qiwen'
OUTPUT_TAG = '_ensemble_consensus_qiwenevents'
multiple_number = 4
T_0 = 75.1
position = int(np.ceil(T_0 * multiple_number))

das = da_system().load(ANALYSIS_DATA_DIR / 'x_analysis_init.pkl')

sv = das.getStateVector()
x_nature = sv.getTrajectory()

obs = das.getObsData()
y_obs = obs.getVal()

Ta = das.acyc_step
fcst_dt = das.fcst_dt

das.multiplicative_inflation = 1.04

print('Regime reference:', REGIME_REFERENCE_MODE)
print('Ambiguous-regime policy:', AMBIGUOUS_REGIME_POLICY)
print('Candidate pool:', CANDIDATE_POOL)

## Load OSSE Histories


In [ ]:
with open(OSSE_DATA_DIR / 'osse_Xa_history.pkl', 'rb') as f:
    osse_Xa_history = pickle.load(f)

with open(OSSE_DATA_DIR / 'osse_summary.pkl', 'rb') as f:
    osse_summary = pickle.load(f)

print('osse_Xa_history shape:', osse_Xa_history.shape)
print("osse_summary['t0']:", osse_summary['t0'])
print("osse_summary['num_cycles']:", osse_summary['num_cycles'])


## Detect Regime Changes


In [ ]:
# Detect every regime transition in the NATURE trajectory as a sign change
# of the x component (a crossing of x=0 between two consecutive time steps).
# With EVENT_SELECTION='qiwen', the function accepts crossings in both
# directions, keeps those occurring after step 500 whose preceding residence
# time is longer than 385 steps, and retains at most the first 100 events.
# The complete list of crossings is also saved in RESULTS_DIR for reuse.
regime_change_info = detect_regime_changes(
    x_nature,
    RESULTS_DIR,
    position=position,
    output_filename='cse_qiwen_xyz_control_change_region_time_ensemble_consensus_qiwenevents.pkl',
    event_selection=EVENT_SELECTION,
)

# All detected x=0 crossings in the NATURE run.
change_region_time = regime_change_info['change_region_time']
# Forecast/control horizon used by the function's diagnostic sign check.
check_position = regime_change_info['check_position']
# Filtered crossing times used as the CSE event set.
t_start_series = regime_change_info['t_start_series']
# Time spent in the previous regime before each selected crossing.
t_start_residence = regime_change_info['t_start_residence']
# Mean interval between all detected crossings, before event filtering.
mean_crossing_interval = regime_change_info['mean_crossing_interval']


## Visual Check of Regime Crossings


In [ ]:
# Plot the x component of the NATURE trajectory and mark the regime-change
# events selected in the previous cell. The upper panel shows a wide view
# of the first 20,000 steps; the lower panel zooms around the first event.
# Green and orange lines identify the two crossing directions (+ to - and
# - to +), while blue lines mark the DA-cycle-aligned time at which control
# would begin: position steps before each selected crossing.
# The function saves the PNG in PathFigures and returns the Matplotlib
# figure, its two axes, and the complete path of the saved image.
fig_regime_crossings, axes_regime_crossings, regime_crossings_path = plot_regime_crossings(
    x_nature,
    t_start_series,
    Ta,
    PathFigures,
    position=position,
    output_filename='regime_crossings_qiwen_qiwenevents.png',
    event_selection=EVENT_SELECTION,
)


## Lorenz Attractor View of Regime-Change Points


In [ ]:
# Show the first 40 DA-cycle-aligned CSE starting points on the Lorenz
# attractor, using one 3D view and the x-y, x-z, and y-z projections.
fig_lorenz_starts, axes_lorenz_starts, lorenz_starts_path = plot_lorenz_attractor_aligned_cse_starts(
    x_nature,
    t_start_series,
    Ta,
    PathFigures,
    position=position,
)
lorenz_starts_tagged_path = lorenz_starts_path.with_name(f'{lorenz_starts_path.stem}{OUTPUT_TAG}{lorenz_starts_path.suffix}')
lorenz_starts_path.replace(lorenz_starts_tagged_path)
lorenz_starts_path = lorenz_starts_tagged_path


## CSE Parameters


In [ ]:
# Configure the single-event CSE diagnostic: D is the norm of each xyz
# control perturbation, experiment_step is the number of DA cycles, and
# EVENT_INDEX selects one event from t_start_series.
D = 0.05
amp = 1.0 / D
experiment_step = 1000
EVENT_INDEX = 1

# Lorenz-63 parameters and integration time step used by the CSE.
solver_params = {
    'model_func': lorenz63,
    'sigma': 10.0,
    'beta': 8.0 / 3.0,
    'rho': 28.0,
    'dt': fcst_dt,
}

print('position:', position)


## Initialize CSE Experiment


In [ ]:
# Select one detected regime-change time and place the CSE start position
# steps before it, rounded down to the nearest DA-cycle boundary (Ta).
t_start = t_start_series[EVENT_INDEX]
t0 = (t_start - position) - (t_start - position) % Ta
# Locate that aligned start time in the saved OSSE analysis history.
osse_start_index = (t0 - osse_summary['t0']) // Ta
if osse_summary['t0'] + osse_start_index * Ta != t0:
    raise ValueError('t0 is not aligned with the OSSE grid')

# Initialize the analysis ensemble and the controlled truth from the same
# absolute time t0.
Xa_initial = np.matrix(osse_Xa_history[osse_start_index])  # (xdim, edim)
x_original = np.asarray(x_nature[:, t0]).ravel().copy()

# Allocate storage for the analysis ensemble at every DA cycle and for the
# controlled truth at every model time step.
Xa_history_control = np.zeros((experiment_step + 1, das.xdim, das.edim))
Xa_history_control[0] = osse_Xa_history[osse_start_index]
evo_history = np.zeros((3, experiment_step * Ta + 1))
evo_history[:, 0] = x_original

# Check the ensemble dimensions and measure its initial mean-state error
# relative to the NATURE state.
assert Xa_initial.shape == (das.xdim, das.edim)
rmse_init = np.linalg.norm(np.asarray(Xa_initial).mean(axis=1) - x_original) / np.sqrt(3)

print('rmse_init:', rmse_init)
print('t0:', t0)
print('osse_start_index:', osse_start_index)
print('np.sign(x_original[0]):', np.sign(x_original[0]))


## Visual Check of Initialization


In [ ]:
# Pre-run visual check of the CSE initialization.
# This confirms that t0 is correctly placed before the selected event and
# shows where the inherited OSSE analysis ensemble sits relative to nature.
fig_cse_init, ax_cse_init, cse_init_path = plot_cse_initialization(
    x_nature=x_nature,
    Xa_initial=Xa_initial,
    t0=t0,
    t_start=t_start,
    osse_start_index=osse_start_index,
    rmse_init=rmse_init,
    path_figures=PathFigures,
    output_filename='cse_initialization_ensemble_consensus_qiwenevents.png',
)


## Run One CSE Event with Ensemble Consensus

At the beginning of every DA cycle, the regime is inferred from the current
analysis ensemble:

- all member values satisfy `x > 0`: positive-regime consensus;
- all member values satisfy `x < 0`: negative-regime consensus;
- otherwise: ambiguous regime.

The usual CSE trigger is retained: control is considered when at least one
member predicts a regime transition. The consensus sign replaces `nature` or
`member0` only where a signed current-regime reference is required. If the
analysis ensemble is ambiguous, this pilot stops and records an information
failure rather than selecting an arbitrary sign. `candidate_pool='full'` is
retained so that the regime-reference rule is the only change relative to
notebook 6.

In [ ]:
# Reload the module so this cell sees the ensemble-consensus implementation.
import importlib
import class_cse_qiwen as _cse_qiwen_module
_cse_qiwen_module = importlib.reload(_cse_qiwen_module)
run_single_cse_core = _cse_qiwen_module.run_single_cse


def run_single_cse(
    event_index,
    D_run,
    seed,
    regime_reference=REGIME_REFERENCE_MODE,
    candidate_pool=CANDIDATE_POOL,
    ambiguous_regime_policy=AMBIGUOUS_REGIME_POLICY,
):
    return run_single_cse_core(
        event_index=event_index,
        D_run=D_run,
        seed=seed,
        das=das,
        x_nature=x_nature,
        t_start_series=t_start_series,
        osse_Xa_history=osse_Xa_history,
        osse_summary=osse_summary,
        experiment_step=experiment_step,
        position=position,
        Ta=Ta,
        fcst_dt=fcst_dt,
        path_figures=PathFigures,
        total_events=40,
        regime_reference=regime_reference,
        candidate_pool=candidate_pool,
        ambiguous_regime_policy=ambiguous_regime_policy,
        control_indices=CONTROL_INDICES,
        T_0=T_0,
        multiple_number=multiple_number,
        cle_target_output_dir=None,
    )


single_cse_result = run_single_cse(
    event_index=EVENT_INDEX,
    D_run=D,
    seed=1,
)

# Export a CLE target only for a completed successful experiment.
if single_cse_result['success']:
    single_cse_result['cle_target_files'] = _cse_qiwen_module.export_cle_target_inputs(
        single_cse_result,
        CLE_TARGET_DATA_DIR,
    )
else:
    single_cse_result['cle_target_files'] = {}

result_file = RESULTS_DIR / (
    f'cse_qiwen_xyz_control_ensemble_consensus_event_{EVENT_INDEX}_D{D}.pkl'
)
temporary_result_file = result_file.with_suffix(result_file.suffix + '.tmp')
with temporary_result_file.open('wb') as handle:
    pickle.dump(single_cse_result, handle)
temporary_result_file.replace(result_file)

print('Event index:', single_cse_result['event_index'])
print('Success:', single_cse_result['success'])
print('Failure cycle:', single_cse_result['failure_cycle'])
print('Failure reason:', single_cse_result['failure_reason'])
print('Completed cycles:', single_cse_result['n_completed'])
print('Control ratio:', single_cse_result['control_ratio'])
print('False alarm ratio:', single_cse_result['false_alarm_ratio'])
print('Success control ratio:', single_cse_result['success_control_ratio'])
print('Branch counts:', single_cse_result['branch_counts'])
print('Safe searches:', single_cse_result['n_safe_searches'])
print('Ambiguous regime cycles:', single_cse_result['ambiguous_regime_cycles'].tolist())
print('Saved result:', result_file)
if single_cse_result['cle_target_files']:
    print('CLE_TARGET input files:')
    for cle_input_name, cle_input_path in single_cse_result['cle_target_files'].items():
        print(f'  {cle_input_name}: {cle_input_path}')

## Analysis-Ensemble Regime Consensus

This diagnostic displays the complete set of current analysis-member values of
`x`. A cycle is unambiguous only when the full ensemble lies strictly on one
side of zero. The diagnostic uses no nature-run values.


In [ ]:
consensus_history = single_cse_result['regime_consensus_history']
consensus_cycles = np.asarray([row['cycle_index'] for row in consensus_history], dtype=int)
consensus_x = np.asarray([row['x_values'] for row in consensus_history], dtype=float)
consensus_sign = np.asarray([row['regime_sign'] for row in consensus_history], dtype=float)

n_positive_consensus = int(np.sum(consensus_sign > 0))
n_negative_consensus = int(np.sum(consensus_sign < 0))
n_ambiguous = int(np.sum(consensus_sign == 0))

fig_consensus, ax_consensus = plt.subplots(figsize=(14, 5.5), constrained_layout=True)
for member in range(consensus_x.shape[1]):
    ax_consensus.plot(
        consensus_cycles,
        consensus_x[:, member],
        marker='o',
        markersize=2.2,
        linewidth=0.9,
        alpha=0.65,
        label=f'analysis member {member}',
    )
ax_consensus.fill_between(
    consensus_cycles,
    consensus_x.min(axis=1),
    consensus_x.max(axis=1),
    color='gray',
    alpha=0.15,
    label='ensemble x range',
)
ambiguous_mask = consensus_sign == 0
if np.any(ambiguous_mask):
    ax_consensus.scatter(
        consensus_cycles[ambiguous_mask],
        np.zeros(np.sum(ambiguous_mask)),
        color='black',
        marker='x',
        s=70,
        linewidths=2,
        label='ambiguous regime',
        zorder=6,
    )
ax_consensus.axhline(0.0, color='black', linestyle='--', linewidth=1.0)
ax_consensus.set_xlabel('DA cycle')
ax_consensus.set_ylabel('analysis-member x')
ax_consensus.set_title('Current Regime Inferred from Whole-Ensemble Agreement')
ax_consensus.grid(alpha=0.25)
ax_consensus.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=4)

consensus_figure_path = PathFigures / 'analysis_ensemble_regime_consensus.png'
fig_consensus.savefig(consensus_figure_path, dpi=200, bbox_inches='tight')
plt.show()

print('Positive-consensus cycles:', n_positive_consensus)
print('Negative-consensus cycles:', n_negative_consensus)
print('Ambiguous cycles:', n_ambiguous)
print('Saved consensus diagnostic:', consensus_figure_path)


## Plot Controlled NR vs Free NR


In [ ]:
# Plot the controlled and free trajectories only when at least one cycle completed.
if single_cse_result['n_completed'] > 0:
    import importlib
    import class_cse as _class_cse_module
    _class_cse_module = importlib.reload(_class_cse_module)
    plot_controlled_vs_free_nature_run = _class_cse_module.plot_controlled_vs_free_nature_run

    fig, axes, controlled_vs_free_path = plot_controlled_vs_free_nature_run(
        evo_history=single_cse_result['evo_history'],
        x_nature=x_nature,
        t0=single_cse_result['t0'],
        n_completed=single_cse_result['n_completed'],
        Ta=Ta,
        y_obs=y_obs,
        yo_history=single_cse_result['yo_history'],
        branch_history=single_cse_result['branch_history'],
        perturbed_time=single_cse_result['perturbed_time'],
        xdim=das.xdim,
        path_figures=PathFigures,
        output_filename='nr_controlled_vs_nr_ensemble_consensus_qiwenevents.png',
    )
else:
    print('Controlled-versus-free plot skipped: no CSE cycle was completed.')

## 3D Plot Controlled NR vs Free NR


In [ ]:
# Plot the 3D comparison only when at least one cycle completed.
if single_cse_result['n_completed'] > 0:
    import importlib
    import class_cse_qiwen as cse_qiwen
    cse_qiwen = importlib.reload(cse_qiwen)

    plot_3d_data = cse_qiwen.plot_3d_nature_run_comparison_with_fallback(
        current_result=single_cse_result,
        x_nature=x_nature,
        Ta=Ta,
        path_figures=PathFigures,
        analysis_data_path=ANALYSIS_DATA_DIR / 'x_analysis_init.pkl',
        results_dir=RESULTS_DIR,
        output_tag=OUTPUT_TAG,
        D=D,
        event_index=EVENT_INDEX,
        experiment_step=experiment_step,
        output_filename='NR_3D_comparison_after_control_ensemble_consensus_qiwenevents.png',
    )

    plot_result = plot_3d_data['plot_result']
    plot_evo_history = plot_3d_data['plot_evo_history']
    plot_t0 = plot_3d_data['plot_t0']
    plot_num_cycles = plot_3d_data['plot_num_cycles']
    x_nature_nocontrol_3d = plot_3d_data['nature_nocontrol']
    x_nature_control_3d = plot_3d_data['nature_control']
    fig_3d_nr = plot_3d_data['fig']
    axes_3d_nr = plot_3d_data['axes']
    nr_3d_path = plot_3d_data['output_path']
else:
    print('Three-dimensional comparison skipped: no CSE cycle was completed.')